In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Grey"
ref_col = "ref"                     # 参考序列列名
cmp_cols = ["x1","x2","x3","x4"]   # 比较序列列名（每列一个序列）

df = pd.read_excel(file_path, sheet_name=sheet_name)
X_ref = df[ref_col].to_numpy(dtype=float)
X_cmp = df[cmp_cols].to_numpy(dtype=float).T  # shape=(n_seq, m)

# ========= 2) 参数模板 =========
params = {
    "rho": 0.5  # 分辨系数(0,1)
}

diff = np.abs(X_cmp - X_ref)
dmin, dmax = diff.min(), diff.max()
coeff = (dmin + params["rho"]*dmax) / (diff + params["rho"]*dmax + 1e-12)
grade = coeff.mean(axis=1)
print(grade)


# 灰色关联

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

灰色关联分析 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

小样本、多指标方案与理想序列接近程度评价。

## 局限性

分辨系数和标准化方式会影响排序。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。

In [ ]:
"""
灰色关联

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "灰色关联.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
INDICATOR_COLUMNS = ["指标1", "指标2"]  # TODO: 请填写[指标列名列表]，说明：数值型并已同向化更佳。
RHO = 0.5  # TODO: 请填写[分辨系数]，说明：0 到 1，常用 0.5。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    X = data[INDICATOR_COLUMNS].astype(float)
    X_norm = (X - X.min()) / (X.max() - X.min())
    reference = X_norm.max(axis=0)
    diff = (X_norm - reference).abs()
    d_min, d_max = diff.min().min(), diff.max().max()
    coeff = (d_min + RHO * d_max) / (diff + RHO * d_max)
    score = coeff.mean(axis=1)
    result = data.copy()
    result["灰色关联度"] = score
    result["排名"] = score.rank(ascending=False, method="min")
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result[[*INDICATOR_COLUMNS, "灰色关联度", "排名"]])


if __name__ == "__main__":
    df = load_data()
    run_model(df)
